# 1.Szükséges importok

A feladat megoldásához pySparkot választottam, az adatok mérete miatt pandassal is megoldható, de ez valós adatokkal skálázhatóbb lehet.

In [0]:
from pyspark.sql import functions as f
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 2.Beolvasás

In [0]:
csv_path = "/Volumes/workspace/default/feladat/aviation_delay.csv"

df = (
    spark.read.format("csv")
    .option("sep", ",")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(csv_path)
)

display(df)

# 3.EDA








##1. Statisztika
Az oszlopok adataiból különböző dolgokat szűrhetünk le:


- a dátumot át kell alakítani, hogy a tanításhoz használható alakban legyen
- a visibility és maintenace_events_last_30d null értékeket tartalmaz, mert azokból levesebb van mint a többiből
- a crew_change_last_minute és crew_status_new_FINAL adatai teljesen megegyeznek (teljesen ugyanazokat az értékeket tartalmazzák), emiatt redundáns mindkettőt megtartani
- a sched_buffer_mins_latest és a turnaround_minutes is megegyezik
- a negatív értékek csak azt jelzik, ha sietett a repülő, ezért nem hibás
- a wind speedben vannak 0 értékek, ez nem feltétlen hiba, jelntheti a szélcsendet
- airport_congestion_index maximuma 115,ez túlzsufoltság lehet, de mivel a harmadik kvartilis 71-nél van, így kiugró érték is lehet
- a passenger_load_factor,runway_utilization maximuma 1, minimuma 0 és 0.2, így nincs benne hiba
- az actual_delay_minutes minimuma 2*10^-4 perc, ez rendkívül kicsi, az adatgenerálás miatt sok ilyen értéket tartalmaz, de el kell dobni a tanításhoz ezt az oszlopot
- fontosnak a previous_leg_delay, airport_congestion_index és az időjárásra vonatkozó változók tűnnek (visibility,wind_speed,thunderstorm_flag)
- az adatok nagyon kiegyensúlyozatlanak, a legtöbb nem késik, ezt majd a tanításnál súlyokkal oldom meg


In [0]:
num_features = [f.name for f in df.schema.fields if str(f.dataType).startswith("Double") or str(f.dataType).startswith("Integer")]

stats_df = df.select(num_features).summary()

display(stats_df)

## 2.A tanításhoz felesleges oszlopok törlése


 Több oszlop van benne, amit a tanításhoz ki kell törölni, mert direkten jelzik ha késés történt, és csak a késés után juthatnánk hozzá az adathoz.
 
- actual_gate_out_time_diff
- final_delay_reason
- actual_delay_minutes
- maintenance_closed_after_pushback
- crew_status_new_FINAL (duplikált értékek miatt)
- sched_buffer_mins_latest (duplikált értékek miatt)


-  delay_over_15m **megmarad**, a célfüggvény miatt

In [0]:
df_clean = df.drop("actual_gate_out_time_diff","final_delay_reason","actual_delay_minutes","maintenance_closed_after_pushback","crew_status_new_FINAL","sched_buffer_mins_latest")

display(df_clean)

## 2. Timestamp szétbontása

A timestamp-et szétbontom külön évre, hónapra, napra, órára, percre, ez a tanításhoz szükséges, illetve a null adatok feltöltéséhez is hasznos.

In [0]:
df_clean2 = df_clean.withColumn("Year", f.year(df_clean.flight_datetime))\
    .withColumn("Month", f.month(df_clean.flight_datetime))\
    .withColumn("Day", f.dayofmonth(df_clean.flight_datetime))\
    .withColumn("Hour",f.hour(df_clean.flight_datetime))\
    .withColumn("Minute",f.minute(df_clean.flight_datetime))
    
display(df_clean2)

## 3. Train és test halmazra bontás

A null értékek feltöltése előtt szétbontom a halmazokat, az adatszivárgás elkerülése érdekében.
Mivel időrendi sorrendben vannak, így nem keverhetem össze az adatokat (mert akkor jövőbeli értékeket is megkap), az első 80% lesz a tanító, a többi 20% pedig a teszt.



In [0]:
df_clean2 = df_clean2.withColumn("unix_time", f.unix_timestamp("flight_datetime"))

cutoff = df_clean2.select("unix_time").stat.approxQuantile("unix_time",[0.8],0.0)[0]

train_df = df_clean2.filter(f.unix_timestamp("flight_datetime") <= cutoff).drop("flight_datetime").drop("unix_time")
test_df  = df_clean2.filter(f.unix_timestamp("flight_datetime") > cutoff).drop("flight_datetime").drop("unix_time")


print(f"Tanító halmaz: {train_df.count()} sor")
print(f"Teszt halmaz: {test_df.count()} sor")

display(train_df)
display(test_df)

## 4.Null értékek kitöltése

Az oszlopok amik nullt tartalmaznak, ki kell tölteni valamilyen értékkel
- visibility - a trainből számított mediánjával töltjük fel, az origin airport, és a hónap alapján (ha nincs, akkor a globális mediánnal a train halmazból)
- maintenance_events_last_30d - A null értékeket 0-val töltöm fel



In [0]:
train_df = train_df.fillna(0,subset=["maintenance_events_last_30d"])
test_df = test_df.fillna(0,subset=["maintenance_events_last_30d"])

global_median = train_df.stat.approxQuantile("visibility", [0.5], 0.01)[0]

monthly_origin_medians = train_df.groupBy("origin", "Month").agg(
    f.expr("percentile_approx(visibility, 0.5)").alias("median_visibility")
)

train_df = train_df.join(monthly_origin_medians, on=["origin", "Month"], how="left")
train_df = train_df.withColumn("visibility", f.coalesce(f.col("visibility"), f.col("median_visibility")))
train_df = train_df.drop("median_visibility")
train_df = train_df.fillna(global_median, subset=["visibility"])


test_df = test_df.join(monthly_origin_medians, on=["origin", "Month"], how="left")
test_df = test_df.withColumn("visibility", f.coalesce(f.col("visibility"), f.col("median_visibility")))
test_df = test_df.drop("median_visibility")
test_df = test_df.fillna(global_median, subset=["visibility"])

display(train_df)
display(test_df)


# 4.Tanítás, tesztelés


## 1.Baseline
Baselineként a megadott predikciós modellt használom, az "ops_delay_prediction_v2" oszlopból, ezzel hasonlítom majd össze a saját modelleket. Kiszámolom az F1-scoret, az accuracyt, illetve kiírom a tévesztési mátrixot.

In [0]:
df_baseline = test_df.withColumn("baseline_pred", f.when(f.col("ops_delay_prediction_v2") > 15, 1.0).otherwise(0.0))
df_baseline = df_baseline.withColumn("delay_over_15m",f.col("delay_over_15m").cast("double"))

evaluator_baseline = MulticlassClassificationEvaluator(labelCol="delay_over_15m", predictionCol="baseline_pred")

f1_baseline = evaluator_baseline.evaluate(df_baseline, {evaluator_baseline.metricName: "f1"})
accuracy_baseline = evaluator_baseline.evaluate(df_baseline, {evaluator_baseline.metricName: "accuracy"})

print(f"Baseline F1-score: {f1_baseline:.4f}")
print(f"Baseline Accuracy: {accuracy_baseline:.4f}")

print("Baseline Tévesztési mátrix:")
df_baseline.crosstab("delay_over_15m", "baseline_pred").show()

baseline_class_1 = evaluator_baseline.evaluate(
    df_baseline, 
    {evaluator_baseline.metricName: "fMeasureByLabel", evaluator_baseline.metricLabel: 1.0}
)

print(f"Valós F1-score (csak a késésekre): {baseline_class_1:.4f}")

## 2.Logisztikus regresszió

Ezt választottam az első fejlettebb modellnek, mert alkalmas két érték közti csoportosításhoz és gyorsan tanítható. Minden futásnál ugyanazt az eredményt adja, így nincs szükség seedre a reprodukálhatóság miatt.

Hátránya viszont, hogy a szöveges adatokhoz One Hot Encodingot kell használni, ami az oszlopok számát megnöveli.

A validációhoz a test halmazon értékeltem ki a modellt, amiben az utolsó 20%-a van az adatoknak időrendi sorrendben.

A tanításban létrehoztam egy oszlopot a súlyoknak, mivel sokkal több a nem késő járat, mint a késő, ezért a késések arányával kapja a hibapontokat.(Ha fals negatív nagyobb büntetést kap).

Ezután a szöveges oszlopokat numerikussá alakítottam One Hot Encodinggal, hogy értelmezni tudja a modell, majd az összes adatot vektorizálom, és skálázom (fontos a logisztikus regressziónál, mert enélkül nem tudja megítélni, melyik adat tényleg fontos a nagyságrend miatt).Végül létrehozom a pipelinet, betanítom a modellt, és tesztelem. Itt is F1-score, accuracy, és tévesztési mátrix szerint értékelem ki.

In [0]:
train_df = train_df.withColumn("label", f.col("delay_over_15m").cast("integer"))
test_df  = test_df.withColumn("label", f.col("delay_over_15m").cast("integer"))


num_negatives = train_df.filter(f.col("label") == 0).count()
num_positives = train_df.filter(f.col("label") == 1).count()

#a súlyok aránya
bal_weights = num_negatives/num_positives

train_df = train_df.withColumn("weight",f.when(f.col("label") == 1, bal_weights).otherwise(1.0))

#szöveges oszlopok
categorical_cols = ["origin", "destination", "aircraft_type"]

indexers = [
    StringIndexer(inputCol=col,outputCol=col+"_index",handleInvalid="keep")
    for col in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[col+"_index" for col in categorical_cols],
    outputCols=[col+"_ohe" for col in categorical_cols],
    handleInvalid="keep"
)
#numerikus oszlopok
num_cols = [
    "scheduled_departure_hour", "route_distance", "aircraft_age", "turnaround_minutes", 
    "passenger_load_factor", "visibility", "wind_speed", "precipitation", 
    "thunderstorm_flag", "airport_congestion_index", "runway_utilization", 
    "gate_availability", "crew_rest_hours", "crew_change_last_minute", 
    "previous_leg_delay", "maintenance_events_last_30d", "ops_delay_prediction_v2", "Year", "Month", "Day", "Hour", "Minute"
]

assembler_inputs = num_cols + [col+"_ohe" for col in categorical_cols]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="unscaled_features",
    handleInvalid="keep"
)
scaler = StandardScaler(
    inputCol="unscaled_features", 
    outputCol="features", 
    withStd=True, 
    withMean=False
)

lr = LogisticRegression(featuresCol="features", labelCol="label",weightCol="weight",maxIter=20)

pipeline_stages = indexers + [encoder, assembler, scaler, lr]
pipeline = Pipeline(stages=pipeline_stages)

print("Logisztikus regresszió tanítása...")
model = pipeline.fit(train_df)
print("Tanítás befejeződött.")

predictions = model.transform(test_df)

evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="prediction")

f1_lr = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
accuracy_lr = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})

print(f"Logisztikus Regresszió F1-score: {f1_lr:.4f}")
print(f"Logisztikus Regresszió Accuracy: {accuracy_lr:.4f}")

print("Logisztikus Regresszió Tévesztési mátrix:")
predictions.crosstab("label", "prediction").show()

f1_class_1 = evaluator.evaluate(
    predictions, 
    {evaluator.metricName: "fMeasureByLabel", evaluator.metricLabel: 1.0}
)

print(f"Valós F1-score (csak a késésekre): {f1_class_1:.4f}")


## 3.Random forest

A második fejlettebb modellhez random forestet válaszottam, egyszerűbb implementálni, nem szükséges a One Hot Encoding, a kiugró értékekre kevésbé érzékeny és az adatokat se kell skálázni. Hogy reprodukálható legyen, a 42-es seedet használom, a fák száma 30, a mélység 6.

Itt egyszerűen vektorizáltam az indexelt oszlopokat, és létrehoztam a modellt (a súlyokat is implementálva),a pipelinet, és a test halmazon értékeltem ki. Ezután az F1-score,accuracy, és tévesztési mátrix alapján értékeltem.


In [0]:
#oszlopok törlése az előző modellből
drop_cols = ["features","rawPrediction","probability","prediction"]
train_df = train_df.drop(*drop_cols)
test_df = test_df.drop(*drop_cols)

assembler_inputs = num_cols + [col+"_index" for col in categorical_cols]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="unscaled_features",
    handleInvalid="keep"
)

rf = RandomForestClassifier(featuresCol='unscaled_features', labelCol='label',weightCol="weight", numTrees=30,maxDepth=6,seed=42)

pipeline_stages = indexers + [assembler,rf]
pipeline_rf = Pipeline(stages=pipeline_stages)

print("Random Forest klasszifikáció tanítása...")
model_rf = pipeline_rf.fit(train_df)
print("Tanítás befejeződött.")

predictions_rf = model_rf.transform(test_df)
evaluator_rf = MulticlassClassificationEvaluator(labelCol="label",metricName="prediction")

accuracy_rf = evaluator_rf.evaluate(predictions_rf,{evaluator_rf.metricName:"accuracy"})
f1_rf = evaluator_rf.evaluate(predictions_rf,{evaluator_rf.metricName:"f1"})

print(f"Random Forest Accuracy (Találati arány): {accuracy_rf:.4f}")
print(f"Random Forest F1-score: {f1_rf:.4f}")

f1_class_1_rf = evaluator_rf.evaluate(
    predictions_rf, 
    {evaluator_rf.metricName: "fMeasureByLabel", evaluator_rf.metricLabel: 1.0}
)

print(f"Valós F1-score (csak a késésekre): {f1_class_1_rf:.4f}")

print("\nRandom Forest Tévesztési mátrix:")
predictions_rf.crosstab("label", "prediction").show()




A logisztikus regressziós modellnél nagyon nehéz volna megtalálni a legfontosabb oszlopokat a One Hot Encoding miatt, ezért ezekhez a random forest modellt használom.

A random forest modell szerint a legjelentősebb oszlopok :
- previous_leg_delay (11.7%), - leginkább az előző járat késésétől függ, hogy a jelenlegi is késni fog-e
- destination_index (9.4%) és origin_index(8.7%) - sokat számít honnan indul a repülő, és hova érkezik, vannak amelyikek hajlamosabbak a késésre
- airport_congestion_index(9.4%) - a zsúfoltság is erősen befolyásolja a járat pontosságát
- ops_delay_prediction_v2 (9.3%) - felhasználja a becsült késést is, de ez nem a legfontosabb, ezért nem csak erre hagyatkozik a modell

In [0]:
rf_model_trained = model_rf.stages[-1]

importances = rf_model_trained.featureImportances.toArray()
importance_data = [(name, float(imp)) for name, imp in zip(assembler_inputs, importances)]

importance_df = spark.createDataFrame(importance_data, ["Feature", "Importance"])
importance_df = importance_df.orderBy(f.desc("Importance"))

print("\nRandom Forest Legfontosabb Változói:")
importance_df.show(15, truncate=False)

# 5.Összefoglaló

|Modell|Accuracy|Valós F1|Valós pozitív|Fals pozitív|
|:--|:--|:--|:--|:--
|Baseline|0.6542|0.0919|21|391
|Logisztikus regresszió|0.7658|0.1459|24|260
|Random forest|0.9258|0.1188|6|50


A baseline modellhez képest mindkét modell jobb eredményeket ért el, mert az eredeti nagyon sok fals pozitív jelzést adott.
Hozzá képest a logisztikus regresszió hárommal több valós pozitív értéket jelzett, és 131-gyel kevesbb fals pozitívot, emiatt a legmagasabb az F1-score.
A random forest kevés valós pozitívat talált, de mivel óvatosabb, így fals pozitív is kevés volt. A valós F1-score nagyon alacsony, ezért csak nagyon biztos esetekben jelez.


A késéseket legfőbbképpen az előző járatok késése okozza, ezután a cél reptér a legfontosabb tényező, harmadikként pedig a repterek zsúfoltsága.

Operációs oldalról a legtöbb fontos tényező nem befolyásolható, de a gate_availability, runway utilization, és crew_change_last_minute igen, jobb logisztikával kevesebb késés elérhető. (kapukon gyorsabb áthaladással, a felszállópálya jobb kihasználásával)

A három modell közül a logisztikus regressziós modell a legalkalmasabb, de ebben se lehetne teljesen megbízni, viszont egy kisegítő metrikaként használható. Sok tényező miatt lehetetlen tökéletes modellt csinálni, ezért inkább csak egy plusz véleményként hasznos.

A limitácók között a legfontosabb, hogy a kiegyensúlyozatlan adatok miatt inkább fals negatívokat fog adni, illetve nem fogja figyelembe venni az emberi hibákat/baleseteket.

Production környezetben gyakran frissítve (újratanítva) lenne érdemes használni, de csak egy szakértő mellett, aki biztosra megy a jelzés pontosságában.

